# Qwen3-Embedding-0.6B — DIMER text embedding tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/qwen3-embedding-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/qwen3-embedding-pipeline/blob/main/tutorials/qwen3_embedding_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Qwen%2FQwen3--Embedding--0.6B-ffcc4d?style=flat)](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) [![Upstream](https://img.shields.io/badge/Upstream-QwenLM%2FQwen3--Embedding-181717?style=flat&logo=github&logoColor=white)](https://github.com/QwenLM/Qwen3-Embedding) [![arXiv](https://img.shields.io/badge/arXiv-2506.05176-b31b1b.svg)](https://arxiv.org/abs/2506.05176)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** text embeddings (1024-d, last-token pooled, L2-normalised, instruction-aware queries) using the pinned `Qwen/Qwen3-Embedding-0.6B` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/qwen3_embedding_pipeline/pipeline.py` at revision `f90e32992282`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3` (~1207 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

At inference each text is tokenised with left padding and truncated at 8,192 tokens, a 0.6 B-parameter Qwen3 decoder encodes it, the hidden state of the **last token** is taken as the text's vector, and the pipeline L2-normalises it to unit length. Queries are prefixed with a task instruction (`Instruct: …\nQuery:`) because the model is instruction-aware; documents are embedded as-is. **Embeddings are representations, not predictions:** nothing is classified, ranked or decided, and there is no label space. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and tokenizer, and the carried pipeline module adds snapshot verification, input validation with named ceilings, the query/document formatting contract, a fixed output contract and the `cosine_similarity`, `validate_inputs` and `evaluation_report` helpers. The default sample is the four sentences from the pinned upstream README; the cosine values shown for them are a qualitative check, not a benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, prepare a small identified set of queries and documents and validate it into an input manifest, embed queries and documents through the public API with the instruction contract, read the vectors correctly (shape, pooling, normalisation, per-text unit), compare a query with two documents by cosine as a qualitative check, produce an evaluation report that is honestly `not-measurable` because an embedding has no intrinsic metric, exercise an optional BYOD path, and export identifiers alongside vectors plus provenance.

**This notebook does not demonstrate:** reranking (see the sibling Qwen3 reranker pipeline), text generation or chat, classification, clustering quality, retrieval evaluation (nDCG/recall need a labelled query–document set), Matryoshka dimension truncation (fixed at 1024 here), or any training.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available. **Precision differs by device:** the pipeline runs float32 on CPU and bfloat16 on CUDA, so cosine values can differ in the second or third decimal between the two. CPU is slow for large corpora but fine for a handful of sentences: the repository's model card records 6.6 s to load and 0.44 s to embed the four default texts on CPU in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the ~1.19 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and NumPy; what a dense vector, a unit norm and cosine similarity are.
- **Data:** the default sample is four short English sentences (two queries, two documents) taken verbatim from the pinned upstream README, written into the notebook as string literals, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file with one document per non-empty line (at most 64 lines, each under 100,000 characters; text beyond 8,192 tokens is truncated and flagged) plus a query typed into the form. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Qwen/Qwen3-Embedding-0.6B` snapshot (~1207 MB) at revision `97b0c614be4d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'qwen3-embedding-pipeline',
    'repository_revision': 'f90e329922825d3b750d480aacc28ae32f7bc49b',
    'embedded_module': 'src/qwen3_embedding_pipeline/pipeline.py',
    'module_sha256': 'b7801ac4b3e89b250e93927dd1e0af4a26351ae40876fc507b304bdf964f177d',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/qwen3_embedding_pipeline/pipeline.py` @ `f90e32992282`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
MODEL_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "qwen3-embedding-0.6b"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Contract from the pinned upstream README ("Transformers Usage"): left padding, last-token pooling,
# L2 normalisation, max_length 8192, and an "Instruct: ...\nQuery:" prefix on queries only.
EMBEDDING_DIM = 1024  # hidden_size in the pinned config.json; 1_Pooling/config.json word_embedding_dimension
MAX_TEXT_TOKENS = 8192  # tokenizer truncation length; the model's context is 32768 but the README uses 8192
MAX_TEXT_CHARS = 100_000  # pre-tokenisation guard so a runaway string is rejected before it is tokenised
MAX_BATCH = 64  # texts per embed() call
DEFAULT_QUERY_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
POOLING = "last_token"
KINDS = ("query", "document")


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_query(query: str, instruction: str = DEFAULT_QUERY_INSTRUCTION) -> str:
    """Upstream `get_detailed_instruct`: queries carry a task instruction, documents do not."""
    return f"Instruct: {instruction}\nQuery:{query}"


def cosine_similarity(a: Sequence[Sequence[float]], b: Sequence[Sequence[float]]) -> list[list[float]]:
    """Cosine similarity matrix between two lists of vectors (no metric: there is no ground truth)."""
    x = np.asarray(a, dtype=np.float32)
    y = np.asarray(b, dtype=np.float32)
    if x.ndim != 2 or y.ndim != 2 or x.shape[1] != y.shape[1]:
        raise ValueError("inputs must be 2-D with the same embedding dimension")
    x = x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    y = y / np.maximum(np.linalg.norm(y, axis=1, keepdims=True), 1e-12)
    return (x @ y.T).tolist()


INPUT_SCHEMA: dict[str, Any] = {
    "input": "sequence of non-empty str; one vector is returned per text, in input order",
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "text_tokens": [1, MAX_TEXT_TOKENS],
    "kind": list(KINDS),
    "embedding_dim": EMBEDDING_DIM,
    "preprocessing": (
        "left-padded tokenisation truncated at MAX_TEXT_TOKENS; kind='query' prepends "
        "'Instruct: <instruction>\\nQuery:'; last-token pooling, then L2 normalisation"
    ),
}


def _check_inputs(texts: Any, kind: str, instruction: str) -> list[str]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the texts as a list."""
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..{MAX_BATCH} items, got {len(texts)}")
    for i, text in enumerate(texts):
        if not isinstance(text, str):
            raise TypeError(f"texts[{i}] must be str, got {type(text).__name__}")
        if not text.strip():
            raise ValueError(f"texts[{i}] is empty")
        if len(text) > MAX_TEXT_CHARS:
            raise ValueError(f"texts[{i}] has {len(text)} chars; ceiling is {MAX_TEXT_CHARS}")
    if kind not in KINDS:
        raise ValueError(f"kind must be one of {KINDS}")
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError("instruction must be a non-empty str")
    return list(texts)


def validate_inputs(
    texts: Sequence[str],
    kind: str = "document",
    instruction: str = DEFAULT_QUERY_INSTRUCTION,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``embed`` would — both route through
    ``_check_inputs`` — so a caller that wants the finding recorded catches the exception and
    stores ``str(exc)`` under ``findings``. Token-level truncation cannot be observed here
    because it happens inside the tokenizer; ``embed`` reports it in ``truncated``.
    """
    checked = _check_inputs(texts, kind, instruction)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"text-{i}", "chars": len(text), "kind": kind}
            for i, text in enumerate(checked)
        ],
        "kind": kind,
        "instruction": instruction if kind == "query" else None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], labels: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Embeddings are representations, so the repository ships no performance metric —
    ``cosine_similarity`` is a comparison helper, not a score against ground truth. The verdict
    is therefore always ``not-measurable`` (EVAL9), including when ``labels`` is supplied:
    the parameter exists for interface parity with the fleet's other pipelines and is recorded
    in ``reason`` rather than scored.
    """
    embeddings = result["embeddings"]
    supplied = labels is not None
    return {
        "task": "text embedding (dense representation, no label space)",
        "score_semantics": (
            f"{EMBEDDING_DIM}-d unit-norm vectors, {POOLING} pooling; cosine between two vectors of "
            "this model is a similarity in [-1, 1], not a probability and not calibrated"
        ),
        "sample_kind": sample_kind,
        "n_texts": len(embeddings),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the output is a representation, not a prediction: the pipeline exposes no performance "
            "metric, only the cosine_similarity comparison helper"
            + (
                "; labels were supplied but no metric helper exists to score them here"
                if supplied
                else ""
            )
        ),
        "needs": (
            "a downstream labelled task: for retrieval, a query-document set with relevance "
            "judgements scored by nDCG@k or recall@k; for classification or clustering, labelled "
            "texts and a fitted classifier or cluster assignment — none of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class Qwen3EmbeddingPipeline:
    """Text embedder. `_runner` maps formatted texts to (pooled un-normalised vectors, token counts)."""

    _runner: Callable[[list[str]], tuple[np.ndarray, list[int]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Qwen3EmbeddingPipeline:
        import torch
        from transformers import AutoModel, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), dict(local_files_only=True)
        elif allow_download:
            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        tokenizer = AutoTokenizer.from_pretrained(
            source, padding_side="left", trust_remote_code=False, **kwargs
        )
        model = AutoModel.from_pretrained(source, dtype=dtype, trust_remote_code=False, **kwargs)
        model = model.to(resolved_device).eval()

        def runner(texts: list[str]) -> tuple[np.ndarray, list[int]]:
            batch = tokenizer(
                texts, padding=True, truncation=True, max_length=MAX_TEXT_TOKENS, return_tensors="pt"
            )
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                hidden = model(**batch).last_hidden_state
            pooled = hidden[:, -1]  # left padding: the last position is the last real token of every row
            counts = batch["attention_mask"].sum(dim=1).tolist()
            return pooled.float().cpu().numpy(), [int(c) for c in counts]

        return cls(runner, resolved_device)

    def _validate(self, texts: Any, kind: str, instruction: str) -> list[str]:
        return _check_inputs(texts, kind, instruction)

    def embed(
        self,
        texts: Sequence[str],
        kind: str = "document",
        instruction: str = DEFAULT_QUERY_INSTRUCTION,
    ) -> dict[str, Any]:
        """Embed up to MAX_BATCH texts. `kind="query"` prepends the instruction; documents get none."""
        texts = self._validate(texts, kind, instruction)
        formatted = [format_query(t, instruction) if kind == "query" else t for t in texts]
        pooled, n_tokens = self._runner(formatted)
        pooled = np.asarray(pooled, dtype=np.float32)
        if pooled.shape != (len(texts), EMBEDDING_DIM):
            raise RuntimeError(f"backend returned {pooled.shape}, expected ({len(texts)}, {EMBEDDING_DIM})")
        normalized = pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)
        return {
            "embeddings": normalized.tolist(),
            "dim": EMBEDDING_DIM,
            "pooling": POOLING,
            "normalized": True,
            "kind": kind,
            "instruction": instruction if kind == "query" else None,
            "n_tokens": list(n_tokens),
            "truncated": [n >= MAX_TEXT_TOKENS for n in n_tokens],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `11`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `97b0c614be4d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Qwen3EmbeddingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-embedding-0.6b",
  "modelId": "Qwen/Qwen3-Embedding-0.6B",
  "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
  "files": [
    {
      "path": "1_Pooling/config.json",
      "bytes": 313,
      "sha256": "37bf193fa101f19101bfad9c31d3eb0f786e247b7b1e5cb7f007d730eed1ddbd"
    },
    {
      "path": "README.md",
      "bytes": 17237,
      "sha256": "c34d9b7e5a267ad3fdd13227a253686bc90844ff4744a2a6a86c7c905e3d06f3"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "b5bf1f51fc45be473a54718cef92448d90a1be001bf9b9a44b8c7f10a19feaa9"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 215,
      "sha256": "10667c72ddb772627bf1780cb7f86af8e2ae0032b8c243c731172064105c6961"
    },
    {
      "path": "generation_config.json",
      "bytes": 117,
      "sha256": "28396d421a2108acce96383f6a7de78008f7f1b17f807958f3c14c51dbfb65fb"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191586416,
      "sha256": "0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd"
    },
    {
      "path": "modules.json",
      "bytes": 349,
      "sha256": "84e40c8e006c9b1d6c122e02cba9b02458120b5fb0c87b746c41e0207cf642cf"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11423705,
      "sha256": "def76fb086971c7867b829c23a26261e38d9d74e02139253b38aeb9df8b4b50a"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207487471
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Qwen3EmbeddingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Prepare the sample texts or optional BYOD

The default sample is **public and bundled in code**: the two queries and two documents from the pinned upstream README (Apache-2.0), each given a stable identifier (`q1`, `q2`, `d1`, `d2`) so every vector and similarity can be mapped back to its text. They exist to show the contract, not to measure anything; the repository's smoke run embedded exactly these and reproduced the README's printed cosine matrix to four decimals. BYOD is optional and disabled by default; when enabled, upload one UTF-8 text file (one document per line) and set `BYOD_QUERY`; documents are identified `d1…dN` in file order. The query instruction (`DEFAULT_QUERY_INSTRUCTION`) is printed because it is part of the query vector: a different instruction produces a different embedding. Look for a dictionary naming the sample kind, the identifiers, character counts and the instruction.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
BYOD_QUERY = 'What is the capital of China?'  # @param {type:"string"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    corpus_name = next(iter(uploaded))
    lines = [line.strip() for line in io.TextIOWrapper(io.BytesIO(uploaded[corpus_name]), encoding='utf-8')]
    documents = [line for line in lines if line]
    queries = [BYOD_QUERY.strip()]
    sample_kind = 'BYOD'
else:
    # Public sample: the pinned upstream README's example queries and documents, as string literals.
    queries = ['What is the capital of China?', 'Explain gravity']
    documents = [
        'The capital of China is Beijing.',
        'Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.',
    ]
    corpus_name = 'upstream_readme_example'
    sample_kind = 'public (pinned upstream README example, bundled as literals)'

query_ids = [f'q{i + 1}' for i in range(len(queries))]
document_ids = [f'd{i + 1}' for i in range(len(documents))]
corpus_sha256 = hashlib.sha256('\n'.join(queries + documents).encode('utf-8')).hexdigest()
print({'sample_kind': sample_kind, 'name': corpus_name, 'query_ids': query_ids, 'document_ids': document_ids, 'chars': {i: len(t) for i, t in zip(query_ids + document_ids, queries + documents, strict=True)}, 'corpus_sha256': corpus_sha256, 'query_instruction': DEFAULT_QUERY_INSTRUCTION})

## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `embed` applies — both route through the same private `_check_inputs` — so type, batch size 1..`MAX_BATCH`, non-empty text, character ceiling `MAX_TEXT_CHARS`, a `kind` in `KINDS` and a non-empty instruction are enforced identically. It returns an **input manifest** naming the schema and ceilings, each input's identifier, character count and kind, and the verdict. Both halves of the corpus are validated: the documents produce the manifest, and the queries — which carry the instruction prefix, so their vectors differ from the same text embedded as a document — are validated the same way and recorded under `query_manifest`. The manifest is written to `outputs/qwen3_embedding_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately oversized batch and records the pipeline's own error message as a finding. Token-level truncation cannot be observed at this stage because it happens inside the tokenizer; the result's `truncated` flags are read in Section 6.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_BATCH': MAX_BATCH, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'EMBEDDING_DIM': EMBEDDING_DIM}})
input_manifest = validate_inputs(documents, 'document', names=document_ids)
input_manifest['query_manifest'] = validate_inputs(queries, 'query', names=query_ids)
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(['probe'] * (MAX_BATCH + 1))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-batch-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/qwen3_embedding_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Embed and read the vectors correctly

`embed(texts, kind, instruction)` returns a dict with `embeddings` — one list per input text, in input order, each of length `dim` (1024) — plus `pooling` (`last_token`), `normalized` (`True`: every vector has unit L2 norm), `kind`, the `instruction` applied (queries only, `None` for documents), `n_tokens` per text, `truncated` flags (a text that hit the 8,192-token ceiling was cut and its vector represents only the kept prefix), and the model identity. The unit of embedding is **one vector per text**; there is no per-token or per-chunk output, and a text longer than the window is not chunked for you. Missing data has no meaning here: empty strings are rejected, not embedded. Below, each query is compared with both documents by cosine as a **qualitative check** that the contract works: the matching document should score higher than the unrelated one. Cosine values are similarities in `[-1, 1]` on this model's geometry, not probabilities and not calibrated; absolute values are not comparable across models, and a threshold for "relevant" is the caller's to set on labelled data. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); float32 (CPU) and bfloat16 (CUDA) differ in the second or third decimal. As recorded in the model card, the repository's CPU smoke on these four texts produced the cosine matrix `[[0.7646, 0.1414], [0.1355, 0.6000]]`, equal to the upstream README's printed values to four decimals; that is one observation on four sentences, not a retrieval score.

In [ ]:
query_result = pipe.embed(queries, kind='query', instruction=DEFAULT_QUERY_INSTRUCTION)
document_result = pipe.embed(documents, kind='document')
query_vectors = np.asarray(query_result['embeddings'], dtype=np.float32)
document_vectors = np.asarray(document_result['embeddings'], dtype=np.float32)
print({'query_shape': query_vectors.shape, 'document_shape': document_vectors.shape, 'dim': document_result['dim'], 'pooling': document_result['pooling'], 'normalized': document_result['normalized'], 'norms': [round(float(v), 4) for v in np.linalg.norm(np.vstack([query_vectors, document_vectors]), axis=1)], 'device': pipe.device})
print({'n_tokens': dict(zip(query_ids + document_ids, query_result['n_tokens'] + document_result['n_tokens'], strict=True)), 'truncated': dict(zip(query_ids + document_ids, query_result['truncated'] + document_result['truncated'], strict=True))})
if any(query_result['truncated'] + document_result['truncated']):
    print('NOTE: at least one text hit MAX_TEXT_TOKENS and was truncated; its vector represents the kept prefix only.')
similarity = cosine_similarity(query_result['embeddings'], document_result['embeddings'])
for query_id, row in zip(query_ids, similarity, strict=True):
    print(query_id, {document_id: round(value, 4) for document_id, value in zip(document_ids, row, strict=True)})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. **No intrinsic metric exists** for an embedding: the vectors are representations, the repository ships `cosine_similarity` as a comparison helper and no metric helper, and so the verdict is always `not-measurable` and the report states what would make the task measurable — for retrieval, a query–document set with relevance judgements scored by nDCG@k or recall@k; for classification or clustering, labelled texts and a fitted classifier or cluster assignment. Supplying labels does not change the verdict, because there is no metric to score them with; the helper records that fact in `reason` instead of inventing a number. The report covers the document embeddings (the queries are representations of the same kind, so which half is scored cannot change a `not-measurable` verdict) and is written to `outputs/qwen3_embedding_evaluation_report.json`.

In [ ]:
report = evaluation_report(document_result, sample_kind=sample_kind)
with open('outputs/qwen3_embedding_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is computed: embeddings are representations; the cosine table above is a qualitative check, and a retrieval or classification score needs labelled data.')

## 8. Export identifiers alongside vectors, and provenance

The vectors are written as CSV (`outputs/qwen3_embedding_vectors.csv`) with one row per text — `id`, `kind`, `n_tokens`, `truncated`, then `e0000…e1023` — so every vector stays attached to its identifier for downstream use. Machine-readable JSON preserves the identified texts, the query instruction, the cosine table keyed by identifier, the per-text token counts and truncation flags, the corpus digest, the input manifest, the evaluation report, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device and the precision implied by it). No credentials are recorded.

In [ ]:
import csv

with open('outputs/qwen3_embedding_vectors.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'kind', 'n_tokens', 'truncated'] + [f'e{i:04d}' for i in range(EMBEDDING_DIM)])
    for kind, ids, result in (('query', query_ids, query_result), ('document', document_ids, document_result)):
        for text_id, vector, n_tokens, truncated in zip(ids, result['embeddings'], result['n_tokens'], result['truncated'], strict=True):
            writer.writerow([text_id, kind, n_tokens, truncated] + [f'{value:.7f}' for value in vector])
payload = {
    'texts': {**dict(zip(query_ids, queries, strict=True)), **dict(zip(document_ids, documents, strict=True))},
    'query_instruction': query_result['instruction'],
    'embedding_contract': {'dim': document_result['dim'], 'pooling': document_result['pooling'], 'normalized': document_result['normalized'], 'unit': 'one vector per text'},
    'n_tokens': dict(zip(query_ids + document_ids, query_result['n_tokens'] + document_result['n_tokens'], strict=True)),
    'truncated': dict(zip(query_ids + document_ids, query_result['truncated'] + document_result['truncated'], strict=True)),
    'cosine_similarity': {query_id: dict(zip(document_ids, row, strict=True)) for query_id, row in zip(query_ids, similarity, strict=True)},
    'vectors_file': 'outputs/qwen3_embedding_vectors.csv',
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'kind': sample_kind, 'name': corpus_name, 'corpus_sha256': corpus_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'precision': 'bfloat16' if pipe.device.startswith('cuda') else 'float32',
    },
}
with open('outputs/qwen3_embedding_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The vectors are representations of the texts in this model's 1024-dimensional space: they predict nothing, carry no labels, and their only meaning is relative — cosine between two vectors from the same model and the same instruction. The cosine table on the default sample shows that the contract works on four short English sentences; it is not a retrieval score, and it must not be generalised to other languages, domains, long documents (truncated at 8,192 tokens), or a different query instruction, which changes the query vectors. Values are uncalibrated similarities, absolute levels are model-specific, and any relevance threshold belongs to the caller and to labelled data. Vectors from the CPU (float32) and CUDA (bfloat16) paths are close but not bitwise equal. The evaluation report is `not-measurable` by construction here, which is the honest verdict for an embedding, not a gap in the notebook. The pipeline provides no reranking, generation, classification, chunking, dimension truncation, or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** change the instruction passed to `embed(..., kind='query', instruction=…)` to a task-specific one (for example a code-search task) and watch the cosine table move; enable `USE_BYOD` with a small corpus file and your own query, then hand-label which lines are relevant to compute recall@k yourself — the first step towards a real retrieval number and the labelled data the evaluation report asks for; embed the same text as `kind='query'` and as `kind='document'` to see how much the instruction prefix shifts a vector.

## References

- Repository README: https://github.com/kurtvalcorza/qwen3-embedding-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/qwen3-embedding-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/qwen3-embedding-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
- Upstream code: https://github.com/QwenLM/Qwen3-Embedding
- Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models (2025): https://arxiv.org/abs/2506.05176